In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import seaborn as sns
import matplotlib.pyplot as plt

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
train_file_path = "../input/house-prices-advanced-regression-techniques/train.csv"
test_file_path = "../input/house-prices-advanced-regression-techniques/test.csv"
dataset_df = pd.read_csv(train_file_path)
print("Full train dataset shape is {}".format(dataset_df.shape))
print("Amount of Categorical Columns: ",len(dataset_df.select_dtypes(include=['object']).columns))
print("Amount of Numerical Columns: ", len(dataset_df.select_dtypes(include=['int64', 'float64']).columns))

In [ ]:
# We'll begin by exploring the data.
print(dataset_df.head(3))
dataset_df.info()

# Find columns with null values, starting from the worst ones
# null_vals = dataset_df.isnull().sum()
# print("Total Rows: ", len(dataset_df))
# print(null_vals[null_vals/len(dataset_df) > 0.7].sort_values(ascending=False))
# print(null_vals[null_vals > 0].sort_values(ascending=False))

# Deal with the missing values, it could be of 3 types: Missing at Random (MAR), Missing Completely at Random (MCAR) and Not Missing at Random (NMAR).
# By looking at the dataset, we can conclude that most of these features are Not Missing at Random (NMAR), because the NULL value has a meaning to it.
# The ones that don't have meaning are: LotFrontage (float64) which looks to be MAR and Electrical (object) which seems to be MCAR.

# Now we just set numerics to 0 and objects to NONE when we see missing values.
cleaned_df = dataset_df.copy()
for col in ['MasVnrArea', 'GarageYrBlt']:
    cleaned_df[col] = cleaned_df[col].fillna(0)
for col in ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'MasVnrType']:
    cleaned_df[col] = cleaned_df[col].fillna('NA')
# Use median based on Neighborhood for LotFrontage, and mode for Electrical
cleaned_df['LotFrontage'] = cleaned_df.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))
cleaned_df['Electrical'] = cleaned_df['Electrical'].fillna(cleaned_df['Electrical'].mode()[0])
print(cleaned_df.head(3))
cleaned_df.info()
# All the data has been cleaned, ready for analysis. 

In [ ]:
# Finding correlation between numeric features
correlation_matrix = cleaned_df.corr(numeric_only=True)

# Get the top 15 features most correlated with SalePrice
top_corr_features = correlation_matrix.nlargest(16, 'SalePrice').index
# top_corr_features = top_corr_features.drop(['SalePrice_log']) #excluding saleprice_log
top_corr_matrix = cleaned_df[top_corr_features].corr()
plt.figure(figsize=(12,10))
sns.heatmap(top_corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths = .5)
print(top_corr_features)
# sns.scatterplot(x=cleaned_df['OverallQual'], y=cleaned_df['SalePrice'])

In [ ]:
# Looking at the distributions of the top 15 numeric features

cleaned_df[top_corr_features].hist(figsize=(12, 10), bins=50)
# We can see that some features could be normalized, so lets do that.

cleaned_df['GrLivArea_log'] = np.log1p(cleaned_df['GrLivArea'])
cleaned_df['GarageArea_log'] = np.log1p(cleaned_df['GarageArea'])
cleaned_df['TotalBsmtSF_log'] = np.log1p(cleaned_df['TotalBsmtSF'])
cleaned_df['1stFlrSF_log'] = np.log1p(cleaned_df['1stFlrSF'])
cleaned_df['BsmtFinSF1_log'] = np.log1p(cleaned_df['BsmtFinSF1'])
cleaned_df['LotFrontage_log'] = np.log1p(cleaned_df['LotFrontage'])
cleaned_df = cleaned_df.drop(['GrLivArea','GarageArea','TotalBsmtSF','1stFlrSF','BsmtFinSF1','LotFrontage'], axis=1)
cleaned_df[['GrLivArea_log','GarageArea_log','TotalBsmtSF_log','1stFlrSF_log','BsmtFinSF1_log','LotFrontage_log']].hist(figsize=(12, 10), bins=50)

# The distribution looks better now, let's move on.

In [ ]:
# Now we will encode categorical features
categorical_features = cleaned_df.select_dtypes(include=['object']).columns
num_features = len(categorical_features)

# First, check the distribution of the features
fig, axes = plt.subplots(nrows=13, ncols=3, figsize=(15, 12 * 5))
# Throw away features that are too skewed to be useful: Utilities, Condition2, RoofMat1, Heating
cleaned_df = cleaned_df.drop(['Utilities', 'Condition2', 'RoofMatl', 'Heating'],axis=1)
categorical_features = cleaned_df.select_dtypes(include=['object']).columns
# Everything else looks fine

# 5. Loop through the features and plot on the grid
#    .flatten() is a helper to easily loop through the 2D grid of axes
for feature, ax in zip(categorical_features, axes.flatten()):
    sns.countplot(data=cleaned_df, x=feature, order=cleaned_df[feature].value_counts().index, ax=ax)
    ax.set_title(f'{feature}')
    ax.tick_params(axis='x', rotation=45)
    ax.set_xlabel('')

# 6. Clean up any empty subplots
for i in range(num_features, len(axes.flatten())):
    axes.flatten()[i].set_visible(False)
plt.tight_layout(pad=2.0)
plt.show()

ordinal_cols = [
    'ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC', 'KitchenQual',
    'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC', 'BsmtExposure', 'GarageFinish',
    'BsmtFinType1', 'BsmtFinType2', 'Functional', 'PavedDrive', 'LotShape', 'LandSlope'
] # Just based on my observation

# print(cleaned_df[ordinal_cols].head(5)) # Finding the mapping for each feature
quality_mapping1 = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'NA': 0, 'None': 0}
quality_mapping2 = {'Gd': 4, 'Av': 3, 'Mn': 2, 'No': 1, 'NA': 0, 'None': 0} #BsmtExposure
quality_mapping3 = {'GLQ': 6, 'ALQ': 5, 'BLQ': 4, 'Rec': 3, 'LwQ': 2, 'Unf': 1, 'NA': 0, 'None': 0} #BsmtFinType1, BsmtFinType2
quality_mapping4 = {'Fin': 3, 'RFn': 2, 'Unf': 1, 'NA': 0, 'None': 0} #GarageFinish
quality_mapping5 = {'Y': 3, 'P': 2, 'N': 1, 'NA': 0, 'None': 0} #PavedDrive
quality_mapping6 = {'Typ': 8, 'Min1': 7, 'Min2': 6,'Mod': 5, 'Maj1': 4, 'Maj2': 3, 'Sev': 2, 'Sal': 1, 'NA': 0, 'None': 0} #Functional
quality_mapping7 = {'Gtl': 3, 'Mod': 2, 'Sev': 1, 'NA': 0, 'None': 0} #LandSlope
quality_mapping8 = {'Reg': 4, 'IR1': 3, 'IR2': 2, 'IR3': 1, 'NA': 0, 'None': 0} #LotShape

# Mapping the nominal features (One Hot Encoding) - The nominal columns are the rest of the categorical columns
nominal_cols = [col for col in categorical_features if col not in ordinal_cols]
encoded_nominal_df = pd.get_dummies(cleaned_df[nominal_cols], drop_first=True)

# Mapping the ordinal features
for col in ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC', 'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC']:
    cleaned_df[col] = cleaned_df[col].map(quality_mapping1)
for col in ['BsmtFinType1', 'BsmtFinType2']:
    cleaned_df[col] = cleaned_df[col].map(quality_mapping3)
cleaned_df['BsmtExposure'] = cleaned_df['BsmtExposure'].map(quality_mapping2)
cleaned_df['GarageFinish'] = cleaned_df['GarageFinish'].map(quality_mapping4)
cleaned_df['PavedDrive'] = cleaned_df['PavedDrive'].map(quality_mapping5)
cleaned_df['Functional'] = cleaned_df['Functional'].map(quality_mapping6)
cleaned_df['LandSlope'] = cleaned_df['LandSlope'].map(quality_mapping7)
cleaned_df['LotShape'] = cleaned_df['LotShape'].map(quality_mapping8)

# Final result
cleaned_df = cleaned_df.drop(nominal_cols, axis=1)
final_train_df = pd.concat([cleaned_df, encoded_nominal_df], axis=1)
print("Original shape:", cleaned_df.shape)
print("Shape after encoding nominal features:", final_train_df.shape)

# Numerical and categorical Features are done, now for the analysis

In [ ]:
# Getting the test.csv ready, exact same as train.csv preparation
# Ran into a problem where the train and test datasets have different columns due to the one hot encoding, so i'll clean them together to ensure nothing bad happens.
train_df = pd.read_csv(train_file_path)

test_df = pd.read_csv(test_file_path)

ntrain = train_df.shape[0]
y_train = final_train_df['SalePrice'] 
combined_df = pd.concat((train_df.drop('SalePrice', axis=1), test_df)).reset_index(drop=True)

cleaned_combined_df = combined_df.copy()
for col in ['MasVnrArea', 'GarageYrBlt', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath', 'GarageCars', 'GarageArea']:
    cleaned_combined_df[col] = cleaned_combined_df[col].fillna(0)
for col in ['PoolQC', 'MiscFeature', 'Alley', 'MSZoning', 'Utilities', 'Exterior1st', 'Exterior2nd', 'KitchenQual', 'Functional', 'SaleType', 'Fence', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'MasVnrType']:
    cleaned_combined_df[col] = cleaned_combined_df[col].fillna('NA')
# Use median based on Neighborhood for LotFrontage, and mode for Electrical
cleaned_combined_df['LotFrontage'] = cleaned_combined_df.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))
cleaned_combined_df['Electrical'] = cleaned_combined_df['Electrical'].fillna(cleaned_combined_df['Electrical'].mode()[0])

cleaned_combined_df['GrLivArea_log'] = np.log1p(cleaned_combined_df['GrLivArea'])
cleaned_combined_df['GarageArea_log'] = np.log1p(cleaned_combined_df['GarageArea'])
cleaned_combined_df['TotalBsmtSF_log'] = np.log1p(cleaned_combined_df['TotalBsmtSF'])
cleaned_combined_df['1stFlrSF_log'] = np.log1p(cleaned_combined_df['1stFlrSF'])
cleaned_combined_df['BsmtFinSF1_log'] = np.log1p(cleaned_combined_df['BsmtFinSF1'])
cleaned_combined_df['LotFrontage_log'] = np.log1p(cleaned_combined_df['LotFrontage'])
cleaned_combined_df = cleaned_combined_df.drop(['GrLivArea','GarageArea','TotalBsmtSF','1stFlrSF','BsmtFinSF1','LotFrontage'], axis=1)

cleaned_combined_df = cleaned_combined_df.drop(['Utilities', 'Condition2', 'RoofMatl', 'Heating'],axis=1)

encoded_nominal_combined_df = pd.get_dummies(cleaned_combined_df[nominal_cols], drop_first=True)
for col in ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC', 'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC']:
    cleaned_combined_df[col] = cleaned_combined_df[col].map(quality_mapping1)
for col in ['BsmtFinType1', 'BsmtFinType2']:
    cleaned_combined_df[col] = cleaned_combined_df[col].map(quality_mapping3)
cleaned_combined_df['BsmtExposure'] = cleaned_combined_df['BsmtExposure'].map(quality_mapping2)
cleaned_combined_df['GarageFinish'] = cleaned_combined_df['GarageFinish'].map(quality_mapping4)
cleaned_combined_df['PavedDrive'] = cleaned_combined_df['PavedDrive'].map(quality_mapping5)
cleaned_combined_df['Functional'] = cleaned_combined_df['Functional'].map(quality_mapping6)
cleaned_combined_df['LandSlope'] = cleaned_combined_df['LandSlope'].map(quality_mapping7)
cleaned_combined_df['LotShape'] = cleaned_combined_df['LotShape'].map(quality_mapping8)

cleaned_combined_df = cleaned_combined_df.drop(nominal_cols, axis=1)
final_combined_df = pd.concat([cleaned_combined_df, encoded_nominal_combined_df], axis=1)
print("Original shape:", cleaned_combined_df.shape)
print("Shape after encoding nominal features:", final_combined_df.shape)

final_train_df = final_combined_df[:ntrain]
final_test_df = final_combined_df[ntrain:]
final_train_df['SalePrice'] = y_train
print("Final train shape:", final_train_df.shape)
print("Final test shape:", final_test_df.shape)

In [ ]:
#sns.displot(cleaned_df['SalePrice'], color='g', bins=50, height=6, aspect=2, kde=True) -> The distribution is skewed, take the log to turn it into a normal distribution
cleaned_df['SalePrice_log'] = np.log1p(cleaned_df['SalePrice'])
sns.displot(cleaned_df['SalePrice_log'], color='g', bins=50, height=6, aspect=2, kde=True)

In [ ]:
from sklearn.ensemble import RandomForestRegressor  # Changed to Regressor
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.impute import SimpleImputer
import numpy as np

# print(list(final_train_df))
# print(list(final_test_df))

# print(list(dataset_df))
# print(list(test_df))

# Use the log-transformed SalePrice for training
y = final_train_df['SalePrice'] 

# Drop the original and log-transformed target variables from the training features
X = final_train_df.drop(['SalePrice'], axis=1)

# The test set has no SalePrice columns, but we need to drop the Id for training
X_test = final_test_df.drop('Id', axis=1, errors='ignore')

# --- Align columns before scaling ---
# Get the columns from the training set (after dropping SalePrice)
train_cols = X.columns
test_cols = X_test.columns

# Find columns in training set but not in test set
missing_in_test = set(train_cols) - set(test_cols)
for c in missing_in_test:
    X_test[c] = 0 # Add missing columns to test set and fill with 0

# Find columns in test set but not in training set
missing_in_train = set(test_cols) - set(train_cols)
for c in missing_in_train:
    X[c] = 0 # Add missing columns to train set and fill with 0

# Ensure the order of columns is the same
X_test = X_test[train_cols]

# --- Scaling ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)

# --- Modeling ---
# Use RandomForestRegressor for this regression task
model = RandomForestRegressor(n_estimators=100,
                              max_depth=8,
                              min_samples_leaf=10,
                              random_state=42)

# Fit the model on the log-transformed target
model.fit(X_scaled, y)

# Predict on the scaled test data
predictions = model.predict(X_test_scaled)

# --- Create Submission File ---
# Transform the predictions back from the log scale to the original scale
# predictions = np.expm1(log_predictions)

# Create the output DataFrame using the correct test data variable
output = pd.DataFrame({'Id': final_test_df.Id, 'SalePrice': predictions})
output.to_csv('submission.csv', index=False)

print('Your submission was successfully saved')

# print(final_test_df.head())
# print(final_train_df.head())
# list(final_train_df.columns)